# CRML Debug Notebook

Runs the `FuzzHarness` for **one** generated CRML file with `keep=True`,
so every intermediate artefact survives for inspection:
- Generated Modelica source (candidate + reference)
- OMC build directory and compiled binary
- Simulation result CSVs

**To target a different file** — change the three variables in the *Target* cell.
All other cells are self-contained.

## 1  Build & start JVM

In [1]:
from pathlib import Path

ROOT = Path(".").resolve().parent  # CRML repo root
from experiments.gradle_jvm import GradleJvm

jvm = GradleJvm(
    project_path=ROOT,
    subproject="experiments",
    subproject_dir="submodules/experiments",
)
jvm.build()
jvm.start()

Running: /home/ubuntu/crml/vol/CRML/gradlew experiments:shadowJar  (cwd=/home/ubuntu/crml/vol/CRML)
> Task :language:generateGrammarSource UP-TO-DATE
> Task :language:compileJava UP-TO-DATE
> Task :util:generateGrammarSource NO-SOURCE
> Task :util:compileJava UP-TO-DATE
> Task :compiler:compileJava UP-TO-DATE
> Task :compiler:processResources UP-TO-DATE
> Task :compiler:classes UP-TO-DATE
> Task :compiler:jar UP-TO-DATE
> Task :experiments:compileJava UP-TO-DATE
> Task :experiments:processResources UP-TO-DATE
> Task :experiments:classes UP-TO-DATE
> Task :language:processResources UP-TO-DATE
> Task :language:classes UP-TO-DATE
> Task :language:jar UP-TO-DATE
> Task :util:processResources NO-SOURCE
> Task :util:classes UP-TO-DATE
> Task :util:jar UP-TO-DATE
> Task :experiments:shadowJar UP-TO-DATE

BUILD SUCCESSFUL in 2s
12 actionable tasks: 12 up-to-date
Consider enabling configuration cache to speed up this build: https://docs.gradle.org/9.1.0/userguide/configuration_cache_enabling.ht

## 2  Target

Edit these three variables to select which generated file to debug.

In [22]:
# ── edit these three variables ───────────────────────────────────────────────
TARGET_LLM     = "qwen3.5_27b"   # subdirectory inside generated/
TARGET_REQ     = "speed"          # requirement identifier (e.g. "temp", "speed")
TARGET_ATTEMPT = 3               # attempt number  (k1→1, k2→2, k3→3)
# ─────────────────────────────────────────────────────────────────────────────

DEBUG_WORK_DIR = Path(
    f"~/crml/vol/data/debug/{TARGET_LLM}_{TARGET_REQ}_k{TARGET_ATTEMPT}"
).expanduser()
DEBUG_WORK_DIR.mkdir(parents=True, exist_ok=True)
print(f"Preserved files will be written to:\n  {DEBUG_WORK_DIR}")

Preserved files will be written to:
  /home/ubuntu/crml/vol/data/debug/qwen3.5_27b_speed_k3


## 3  Resolve row & load mapping

In [23]:
from experiments.loader import load_generated

df = load_generated()
mask = (
    (df["llm"]         == TARGET_LLM) &
    (df["requirement"] == TARGET_REQ) &
    (df["attempt"]     == TARGET_ATTEMPT)
)
hits = df[mask]
if hits.empty:
    raise ValueError(
        f"No match for {TARGET_LLM=}, {TARGET_REQ=}, {TARGET_ATTEMPT=}.\n"
        f"Available: {df[['llm','requirement','attempt']].to_string(index=False)}"
    )
row = hits.iloc[0]
print(f"File  : {row['crml']}")
print(f"Domain: {row['domain']}")

File  : /home/ubuntu/crml/vol/CRML/LLM/generated/qwen3.5_27b/SRI_speed_k3.crml
Domain: SRI


In [24]:
from experiments.harness import RequirementMapping
from experiments.batch_run import DEFAULT_REGISTRY

crml_path    = Path(row["crml"])
mapping_path = crml_path.with_name(crml_path.stem + "_mapping.json")

if not mapping_path.exists():
    raise FileNotFoundError(f"No mapping file found at {mapping_path}")

mapping = RequirementMapping.load(mapping_path)
print(mapping.report())

Requirement mapping:
  R1_T                 → (missing)
  R2_T                 → (missing)
  R_T                  → (missing)
  R_speed_all          → (missing)
  R_flow_all           → (missing)

0/5 requirements matched.


In [25]:
entry          = DEFAULT_REGISTRY[row["domain"]]
adapted_domain = mapping.apply_to_domain(entry.domain_spec)

print("Matched output signals:")
for sig in adapted_domain.outputs:
    print(f"  ref={sig.name}  candidate={sig.candidate_name}")

Matched output signals:


## 4  Run harness with `keep=True`

All intermediate files are preserved in `DEBUG_WORK_DIR`.

In [26]:
from experiments.harness import CRMLCompiler, FuzzHarness

compiler = CRMLCompiler()
harness  = FuzzHarness(adapted_domain, compiler, entry.crmltomodelica_path)

candidate_crml = crml_path.read_text()
ref_crml       = entry.ref_crml_path.read_text()

result = harness.run(
    candidate_crml=candidate_crml,
    reference_crml=ref_crml,
    n_iters=20,
    seed=42,
    verbose=True,
    work_dir=DEBUG_WORK_DIR,
    keep=True,
)

print("\n" + result.summary())

[harness] Compiling candidate...
2026-06-01 14:05:00 ERROR crmlVisitorImpl:632 - LOOKUP FAIL: uc.name=[ensure] op=['ensure'] keys=[]


org.antlr.v4.runtime.misc.ParseCancellationException: org.antlr.v4.runtime.misc.ParseCancellationException: no definition found : ('during'true)'ensure'(v<=6.0)


## 5  Inspect preserved files

In [7]:
print(f"Files in {DEBUG_WORK_DIR}:\n")
for p in sorted(DEBUG_WORK_DIR.rglob("*")):
    if p.is_file():
        rel   = p.relative_to(DEBUG_WORK_DIR)
        size  = p.stat().st_size
        print(f"  {str(rel):<60s}  {size:>10,} B")

Files in /home/ubuntu/crml/vol/data/debug/qwen3.5_27b_temp_k3:



## 6  Replay a single failing scenario

Re-runs one failure in isolation so you can narrow down which signal diverges
and when.  The replay artefacts land in `DEBUG_WORK_DIR/replay/`.

In [8]:
if not result.failures:
    print("No failures to replay — all runs passed.")
else:
    first = result.failures[0]
    print(f"First failure:")
    print(f"  params  : {first.params}")
    print(f"  signals : {first.mismatch_signals}")
    print()

    # Named scenarios are stored as {"scenario": name}; resolve to float params.
    if list(first.params.keys()) == ["scenario"]:
        scenario_name = first.params["scenario"]
        params = adapted_domain.scenarios.get(scenario_name, {})
        print(f"Resolved named scenario '{scenario_name}' → {params}")
    else:
        params = first.params

    replay_dir = DEBUG_WORK_DIR / "replay"
    replay_dir.mkdir(exist_ok=True)

    replay = harness.run_scenario(
        candidate_crml=candidate_crml,
        reference_crml=ref_crml,
        params=params,
        verbose=True,
        work_dir=replay_dir,
        keep=True,
    )
    print("\n" + replay.summary())

    print(f"\nReplay files in {replay_dir}:")
    for p in sorted(replay_dir.rglob("*")):
        if p.is_file():
            print(f"  {p.relative_to(replay_dir)}")

NameError: name 'result' is not defined

## Shutdown

In [ ]:
jvm.shutdown()